# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
import os
import duckdb
import pandas as pd
from huggingface_hub import hf_hub_download
from IPython.display import display, Markdown

display(Markdown("""
# 1. Ranked Actions + Reason Codes

The action queue prioritizes content using the Week-4 baseline signals.

The score is used for **decision-support**, not automatic publishing or
content changes.

The reason code explains the main signal behind the recommendation so that a
human reviewer can understand why an item appears in the queue.
"""))

# Load the warehouse sample
parquet_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance_sample.parquet"
)

con = duckdb.connect()

df = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    ga4_engaged_sessions
FROM read_parquet('{parquet_file}')
WHERE gsc_data_available IS TRUE
  AND report_date BETWEEN DATE '2026-06-01' AND DATE '2026-06-30'
""").df()

# Remove duplicate observations at the content/client/date grain
df = df.drop_duplicates(
    subset=["report_date", "client_hash_id", "content_hash_id"]
).copy()

# Baseline action score
df["action_score"] = (
    df["gsc_impressions"].fillna(0) * 0.40 +
    df["gsc_clicks"].fillna(0) * 0.20 +
    df["ga4_sessions"].fillna(0) * 0.20 +
    df["ga4_engaged_sessions"].fillna(0) * 0.10 +
    df["gsc_avg_position"].fillna(999).apply(
        lambda x: 10 if 1 <= x <= 10 else (5 if 11 <= x <= 20 else 0)
    )
)

# One human-readable reason code
def get_reason(row):
    position = row["gsc_avg_position"]
    impressions = row["gsc_impressions"]

    if pd.notna(position) and 1 <= position <= 10:
        return "GOOD_POSITION"
    elif pd.notna(position) and 11 <= position <= 20:
        return "MID_POSITION"
    elif impressions > 0:
        return "LOW_VISIBILITY"
    else:
        return "INSUFFICIENT_SIGNAL"

df["reason_code"] = df.apply(get_reason, axis=1)

# Action mapping
action_map = {
    "GOOD_POSITION": "Optimize Existing Content",
    "MID_POSITION": "Review and Improve",
    "LOW_VISIBILITY": "Investigate Content",
    "INSUFFICIENT_SIGNAL": "Manual Review"
}

df["action_label"] = df["reason_code"].map(action_map)

# Ranked queue
queue = (
    df.sort_values("action_score", ascending=False)
      .reset_index(drop=True)
)

queue["rank"] = queue.index + 1

display(
    queue[
        [
            "rank",
            "report_date",
            "client_hash_id",
            "content_hash_id",
            "action_score",
            "reason_code",
            "action_label"
        ]
    ].head(10)
)


# 1. Ranked Actions + Reason Codes

The action queue prioritizes content using the Week-4 baseline signals.

The score is used for **decision-support**, not automatic publishing or
content changes.

The reason code explains the main signal behind the recommendation so that a
human reviewer can understand why an item appears in the queue.


,rank,report_date,client_hash_id,content_hash_id,action_score,reason_code,action_label
0,1,2026-06-11,client_e547b89c05043229,content_963de14b1f58978f,98655.9,GOOD_POSITION,Optimize Existing Content
1,2,2026-06-29,client_e547b89c05043229,content_eadb33b5df496f4a,19813.4,GOOD_POSITION,Optimize Existing Content
2,3,2026-06-30,client_e547b89c05043229,content_eadb33b5df496f4a,19663.9,GOOD_POSITION,Optimize Existing Content
3,4,2026-06-26,client_e547b89c05043229,content_545bb6cc7081ded3,19624.7,GOOD_POSITION,Optimize Existing Content
4,5,2026-06-12,client_e547b89c05043229,content_963de14b1f58978f,18544.8,GOOD_POSITION,Optimize Existing Content
5,6,2026-06-30,client_06d356715a8ff3b6,content_f88878f155e4838d,18514.4,GOOD_POSITION,Optimize Existing Content
6,7,2026-06-25,client_e547b89c05043229,content_545bb6cc7081ded3,17032.8,GOOD_POSITION,Optimize Existing Content
7,8,2026-06-27,client_e547b89c05043229,content_545bb6cc7081ded3,16468.3,GOOD_POSITION,Optimize Existing Content
8,9,2026-06-29,client_06d356715a8ff3b6,content_f88878f155e4838d,14372.0,GOOD_POSITION,Optimize Existing Content
9,10,2026-06-24,client_e547b89c05043229,content_0ec99ef7d7e11565,12286.6,GOOD_POSITION,Optimize Existing Content


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [2]:
from IPython.display import Markdown, display

display(Markdown("""
# 2. Intended Use and Limits

## Intended use

The playbook is intended for **content/SEO teams** to prioritize which
content pages deserve human review first.

The ranked queue is a **decision-support tool**. It helps reviewers focus
their attention on pages showing stronger measurable search and engagement
signals.

The output should support questions such as:

- Which pages should we inspect first?
- What signal caused this page to be prioritized?
- What action should a reviewer consider?

## Limits

The score does **not** prove that a page needs a refresh or that a particular
action will improve future performance.

The current ranking is based on observed search and engagement signals and
the Week-4 rule-based scoring logic.

The W05 model also measured agreement with the baseline rather than a verified
future outcome.

Therefore, the queue should be treated as **directional decision-support**,
not as a production recommendation engine.

It should not be used to automatically:

- publish or modify content,
- delete content,
- change SEO strategy without review,
- guarantee traffic or ranking improvements,
- make high-impact business decisions without additional evidence.

A human reviewer must consider the page context, business goals, seasonality,
recent changes, and other information that is not represented in the score.
"""))


# 2. Intended Use and Limits

## Intended use

The playbook is intended for **content/SEO teams** to prioritize which
content pages deserve human review first.

The ranked queue is a **decision-support tool**. It helps reviewers focus
their attention on pages showing stronger measurable search and engagement
signals.

The output should support questions such as:

- Which pages should we inspect first?
- What signal caused this page to be prioritized?
- What action should a reviewer consider?

## Limits

The score does **not** prove that a page needs a refresh or that a particular
action will improve future performance.

The current ranking is based on observed search and engagement signals and
the Week-4 rule-based scoring logic.

The W05 model also measured agreement with the baseline rather than a verified
future outcome.

Therefore, the queue should be treated as **directional decision-support**,
not as a production recommendation engine.

It should not be used to automatically:

- publish or modify content,
- delete content,
- change SEO strategy without review,
- guarantee traffic or ranking improvements,
- make high-impact business decisions without additional evidence.

A human reviewer must consider the page context, business goals, seasonality,
recent changes, and other information that is not represented in the score.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [3]:
from IPython.display import Markdown, display

display(Markdown("""
# 3. Human Review + No-Go List

## Human review rules

Every ranked recommendation should be reviewed by a person before action.

The reviewer should check:

1. **Current page context**
   - Is the page still relevant?
   - Does the content satisfy the user's search intent?

2. **Recent changes**
   - Was the page recently updated?
   - Could a recent change explain the current performance?

3. **Search performance**
   - Is the observed position and impression volume stable?
   - Could seasonality or a temporary ranking change explain the signal?

4. **Content quality**
   - Is the information accurate and useful?
   - Are there obvious content gaps or outdated sections?

5. **Business context**
   - Is this page strategically important?
   - Is there a reason not to modify the page despite its ranking?

The reason code should be treated as a **starting point for investigation**, not
as a final diagnosis.

## No-go list

The following actions should **not** be automated by this playbook:

- Automatically rewriting or publishing content.
- Automatically deleting or redirecting pages.
- Automatically changing SEO strategy.
- Automatically declaring a page successful or unsuccessful.
- Automatically making high-impact client decisions.
- Automatically treating a high score as proof that a refresh will improve
  future performance.

The system should recommend **where human attention may be useful**, while the
final decision remains with the reviewer.
"""))


# 3. Human Review + No-Go List

## Human review rules

Every ranked recommendation should be reviewed by a person before action.

The reviewer should check:

1. **Current page context**
   - Is the page still relevant?
   - Does the content satisfy the user's search intent?

2. **Recent changes**
   - Was the page recently updated?
   - Could a recent change explain the current performance?

3. **Search performance**
   - Is the observed position and impression volume stable?
   - Could seasonality or a temporary ranking change explain the signal?

4. **Content quality**
   - Is the information accurate and useful?
   - Are there obvious content gaps or outdated sections?

5. **Business context**
   - Is this page strategically important?
   - Is there a reason not to modify the page despite its ranking?

The reason code should be treated as a **starting point for investigation**, not
as a final diagnosis.

## No-go list

The following actions should **not** be automated by this playbook:

- Automatically rewriting or publishing content.
- Automatically deleting or redirecting pages.
- Automatically changing SEO strategy.
- Automatically declaring a page successful or unsuccessful.
- Automatically making high-impact client decisions.
- Automatically treating a high score as proof that a refresh will improve
  future performance.

The system should recommend **where human attention may be useful**, while the
final decision remains with the reviewer.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [4]:
from IPython.display import Markdown, display

display(Markdown("""
# 4. Monitoring / Retrain Triggers

The playbook should be monitored because search and engagement patterns can
change over time.

## Monitoring signals

The following should be checked periodically:

- Distribution of `gsc_impressions`
- Distribution of `gsc_clicks`
- Distribution of `gsc_avg_position`
- Distribution of `ga4_sessions`
- Distribution of `ga4_engaged_sessions`
- Number of pages receiving each reason code
- Changes in the composition of the ranked queue

## Trigger for investigation

A review should be triggered if:

- Feature distributions change substantially from the data used to build the
  current score/model.
- The ranked queue becomes dominated by one reason code.
- Important signals become unavailable or unusually sparse.
- The relationship between the signals and observed content performance
  changes materially.
- Human reviewers repeatedly disagree with the recommendations.

## Retrain / rebuild trigger

A model or scoring rule should be reconsidered when monitoring shows sustained
change rather than a single unusual observation.

A rebuild should also be considered when a verified future-window outcome
becomes available and shows that the current ranking does not perform as
expected.

These are **monitoring and investigation triggers**, not automatic retraining
rules.

The current project is a research and decision-support workflow, so retraining
should remain a human-controlled process.
"""))


# 4. Monitoring / Retrain Triggers

The playbook should be monitored because search and engagement patterns can
change over time.

## Monitoring signals

The following should be checked periodically:

- Distribution of `gsc_impressions`
- Distribution of `gsc_clicks`
- Distribution of `gsc_avg_position`
- Distribution of `ga4_sessions`
- Distribution of `ga4_engaged_sessions`
- Number of pages receiving each reason code
- Changes in the composition of the ranked queue

## Trigger for investigation

A review should be triggered if:

- Feature distributions change substantially from the data used to build the
  current score/model.
- The ranked queue becomes dominated by one reason code.
- Important signals become unavailable or unusually sparse.
- The relationship between the signals and observed content performance
  changes materially.
- Human reviewers repeatedly disagree with the recommendations.

## Retrain / rebuild trigger

A model or scoring rule should be reconsidered when monitoring shows sustained
change rather than a single unusual observation.

A rebuild should also be considered when a verified future-window outcome
becomes available and shows that the current ranking does not perform as
expected.

These are **monitoring and investigation triggers**, not automatic retraining
rules.

The current project is a research and decision-support workflow, so retraining
should remain a human-controlled process.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [5]:
import os
from IPython.display import Markdown, display

# Create outputs directory
os.makedirs("../outputs", exist_ok=True)

# Export the ranked action queue
output_path = "../outputs/action_playbook_queue.csv"

queue.to_csv(output_path, index=False)

display(Markdown(f"""
# 5. Exports for the Paper

The ranked action queue has been exported for reuse in the research paper.

**File:** `{output_path}`

**Rows exported:** {len(queue):,}

The CSV is generated by the notebook and should remain reproducible rather
than being treated as a manually maintained dataset.
"""))

print("Queue written to:", output_path)
print("Rows written:", len(queue))


# 5. Exports for the Paper

The ranked action queue has been exported for reuse in the research paper.

**File:** `../outputs/action_playbook_queue.csv`

**Rows exported:** 3,876,892

The CSV is generated by the notebook and should remain reproducible rather
than being treated as a manually maintained dataset.


Queue written to: ../outputs/action_playbook_queue.csv
Rows written: 3876892


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.